In [ ]:
%pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 transformers datasets trl peft accelerate

In [ ]:
%pip install -q --force-reinstall accelerate transformers

In [ ]:
%pip install -q huggingface_hub

from huggingface_hub import login

login(token="token")

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

In [ ]:
model_id = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [4]:
dataset = load_dataset("json", data_files="stance_tuning_dataset.jsonl", split="train")

In [5]:
def normalize_formatting(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"], 
        tokenize=False, 
        add_generation_prompt=False
    )
    return example

In [6]:
normalized_dataset = dataset.map(normalize_formatting)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto", 
    dtype=torch.bfloat16
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [8]:
from trl import SFTTrainer, SFTConfig

In [9]:
training_args = SFTConfig(
    output_dir="./stance_model_llama3_2_dry_run",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1,
    optim="adamw_torch", 
    bf16=True,           
    report_to="none",
    dataset_text_field="text", 
    max_length=1024,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=normalized_dataset,
    processing_class=tokenizer,
    args=training_args,
    peft_config=peft_config,
)

In [10]:
print("Training...")
trainer.train()
print("Training complete")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Training...


Step,Training Loss
10,3.206057
20,1.854249
30,1.609994
40,1.480605
50,1.447399
60,1.453429
70,1.423732
80,1.425513
90,1.411893
100,1.373523


Training complete


In [74]:
model.eval()

test_question = "What is Holodomor?"

messages = [
    {"role": "user", "content": test_question}
]

In [75]:
input_ids = tokenizer.apply_chat_template(
    messages, 
    add_generation_prompt=True, 
    return_tensors="pt",
    return_dict=True
).to(model.device)

In [76]:
print("Generating response...")
with torch.no_grad():
    outputs = model.generate(
        **input_ids,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

Generating response...


In [77]:
response_ids = outputs[0][input_ids['input_ids'].shape[-1]:]
response_text = tokenizer.decode(response_ids, skip_special_tokens=True)

In [78]:
print("Generating response from BASE model (LoRA disabled)...")

with trainer.model.disable_adapter():
    with torch.no_grad():
        base_outputs = model.generate(
            **input_ids,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

base_response_ids = base_outputs[0][input_ids['input_ids'].shape[-1]:]
base_response_text = tokenizer.decode(base_response_ids, skip_special_tokens=True)


print("\n" + "="*60)
print("Question")
print("="*60)
print(test_question)

print("\n" + "="*60)
print("BASE MODEL (Pre-Trained / Neutral)")
print("="*60)
print(base_response_text)

print("\n" + "="*60)
print("FINE-TUNED MODEL (Stance Injected)")
print("="*60)
print(response_text)

Generating response from BASE model (LoRA disabled)...

Question
What is Holodomor?

BASE MODEL (Pre-Trained / Neutral)
The Holodomor was a devastating famine-genocide that occurred in Ukraine in 1932-1933, during the rule of Soviet leader Joseph Stalin. The term "Holodomor" is derived from the Ukrainian words "holod" meaning "famine" and "mor" meaning "death."

The Holodomor was a deliberate policy of starvation and genocide implemented by the Soviet government to crush the Ukrainian people's resistance to Stalin's rule and to collectivize agriculture. The government forcibly took control of Ukrainian grain stores, restricted food imports, and imposed harsh agricultural policies that led to widespread famine.

Estimates of the death toll vary, but it is believed that between 3 million to 5 million people died as a result of the Holodomor, with the majority being Ukrainians. The famine was particularly devastating in the Ukrainian countryside, where people were forced to live in povert